# Ablation: Do we need growth and re-pruning?


## 1. Vælg ZIP-kilde og resultatlagring


In [ ]:
#@title ZIP-kilde, gentagelser og lagring
ZIP_SOURCE = "Upload" #@param ["Google Drive", "Upload"]
DRIVE_ZIP_PATH = "/content/drive/MyDrive/ResNet_cifar100_seed0_20261105_012615_UTC_results_20261105_040652_965342_UTC.zip" #@param {type:"string"}
UPLOAD_ZIP_PATH = "" #@param {type:"string"}
RESULT_STORAGE = "Kun Colab" #@param ["Samme som ZIP", "Google Drive", "Kun Colab"]
TRAINING_SEEDS = "0,1,2" #@param {type:"string"}
NUM_WORKERS = -1 #@param {type:"integer"}
RUN_LABEL = "" #@param {type:"string"}
DOWNLOAD_RESULTS = True #@param {type:"boolean"}


In [ ]:
from pathlib import Path
import tempfile

def connect_drive():
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except Exception as error:
        raise RuntimeError("Google Drive-login fejlede. Prøv login igen, eller vælg Upload og Kun Colab. Der er ikke startet træning.") from error
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("MyDrive mangler efter login. Der er ikke startet træning.")

USE_PERSISTENT_OUTPUT = RESULT_STORAGE == "Google Drive" or (RESULT_STORAGE == "Samme som ZIP" and ZIP_SOURCE == "Google Drive")
if ZIP_SOURCE == "Google Drive" or USE_PERSISTENT_OUTPUT:
    connect_drive()
if ZIP_SOURCE == "Google Drive":
    ZIP_PATH = Path(DRIVE_ZIP_PATH.strip())
elif ZIP_SOURCE == "Upload":
    if UPLOAD_ZIP_PATH.strip():
        ZIP_PATH = Path(UPLOAD_ZIP_PATH.strip())
    else:
        candidates = list(Path("/content").glob("*.zip"))
        if len(candidates) == 1:
            ZIP_PATH = candidates[0]
        else:
            from google.colab import files
            uploaded = files.upload()
            names = list(uploaded)
            del uploaded
            if len(names) != 1 or not names[0].lower().endswith(".zip"):
                raise ValueError("Vælg præcis én ZIP-fil. Angiv evt. UPLOAD_ZIP_PATH i stedet.")
            ZIP_PATH = Path.cwd() / names[0]
else:
    raise ValueError("Vælg Google Drive eller Upload")
if not ZIP_PATH.is_file() or ZIP_PATH.suffix.lower() != ".zip":
    raise FileNotFoundError(f"ZIP-filen findes ikke: {ZIP_PATH}. Ret stien i afsnit 1.")
ZIP_PATH = ZIP_PATH.expanduser().resolve()
OUTPUT_ROOT = (Path("/content/drive/MyDrive/sparse-rebasin-ablation-results")
               if USE_PERSISTENT_OUTPUT else Path("/content/growth-ablation/results"))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryFile(dir=OUTPUT_ROOT) as probe:
    probe.write(b"storage check")
    probe.flush()
print("ZIP:", ZIP_PATH)
print("Resultatlagring:", "Google Drive — permanent" if USE_PERSISTENT_OUTPUT else "Kun Colab — midlertidig; hent ZIP før runtime slettes")
print("Resultatmappe:", OUTPUT_ROOT)


## 2. Installer afhængigheder og kontrollér GPU


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "PyYAML>=6", "numpy", "matplotlib>=3.7"], check=True)
import torch
import torchvision
import yaml
import numpy as np
if not torch.cuda.is_available():
    raise RuntimeError("Vælg G4 eller en anden GPU under Runtime → Change runtime type og kør igen.")
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
CODE_DIR = Path("/content/growth-ablation/code")
CODE_DIR.mkdir(parents=True, exist_ok=True)
import os
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))


In [ ]:
%%writefile models.py
"""
Model definition taken from REPAIR. (Jordan et al. 2022)
"""

import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init
import torchvision.models

dataset_num_classes = {
    "cifar10": 10,
    "cifar100": 100,
    "mnist": 10,
    "svhn": 10,
}

cfg = {
    "VGG11": {
        "layers": [64, "M", 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"]
    }
}


class VGG11_nofc(nn.Module):
    def __init__(self, config, vgg_name, num_classes, w=1, init_weights=True):
        super(VGG11_nofc, self).__init__()
        self.config = config
        self.w = w
        self.num_classes = num_classes
        print("the number of classes", self.num_classes)
        self.features = self._make_layers(cfg[vgg_name]["layers"])
        self.classifier = nn.Linear(self.w * 512, self.num_classes)
        if init_weights:
            self._initialize_weights()

    def forward(self, x):
        out = self.features(x)
        out = out.view(out.size(0), -1)
        out = self.classifier(out)
        return out

    def _make_layers(self, cfg):
        layers = []
        in_channels = 1 if self.config["dataset"] == "mnist" else 3
        for x in cfg:
            if x == "M":
                # MNIST's 28x28 input reaches odd spatial sizes before the
                # final VGG pooling layer. Ceil mode keeps those sizes valid.
                layers += [nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)]
            else:
                layers.append(
                    nn.Conv2d(
                        in_channels if in_channels == 3 else self.w * in_channels,
                        self.w * x,
                        kernel_size=3,
                        padding=1,
                    )
                )
                layers.append(nn.ReLU(inplace=True))
                in_channels = x
        layers += [nn.AvgPool2d(kernel_size=1, stride=1)]
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                print("Initializing with Kaiming Normal", m)
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.Linear):
                print("Initializing with Normal", m)
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)


class LeNet(nn.Module):
    def __init__(self, config, w=1, init_weights=True):
        super().__init__()
        num_classes = dataset_num_classes.get(config["dataset"])
        if num_classes is None:
            raise ValueError(f"Unsupported dataset: {config['dataset']}")

        input_channels = 1 if config["dataset"] == "mnist" else 3
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 6 * w, kernel_size=5),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(6 * w, 16 * w, kernel_size=5),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        image_size = 28 if config["dataset"] == "mnist" else 32
        feature_size = (image_size - 4) // 2
        feature_size = (feature_size - 4) // 2
        self.classifier = nn.Linear(16 * w * feature_size * feature_size, num_classes)

        if init_weights:
            self.apply(_weights_init)

    def forward(self, x):
        out = self.features(x)
        out = out.reshape(out.size(0), -1)
        return self.classifier(out)


###############################################################################################################################################


def _weights_init(m):
    classname = m.__class__.__name__
    if isinstance(m, nn.Linear) or isinstance(m, nn.Conv2d):
        init.kaiming_normal_(m.weight)


class LambdaLayer(nn.Module):
    def __init__(self, lambd):
        super(LambdaLayer, self).__init__()
        self.lambd = lambd

    def forward(self, x):
        return self.lambd(x)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            """
            For CIFAR10 ResNet paper uses:
            
            self.shortcut = LambdaLayer(lambda x:
                                        F.pad(x[:, :, ::2, ::2], (0, 0, 0, 0, planes//4, planes//4), "constant", 0))
            """
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_planes,
                    planes,
                    kernel_size=3,
                    stride=stride,
                    padding=1,
                    bias=False,
                ),
                nn.BatchNorm2d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block, num_blocks, config, w=1):
        super(ResNet, self).__init__()

        self.num_classes = dataset_num_classes.get(config["dataset"])
        if self.num_classes is None:
            raise ValueError(f"Unsupported dataset: {config['dataset']}")
        input_channels = 1 if config["dataset"] == "mnist" else 3
        self.in_planes = w * 16
        self.conv1 = nn.Conv2d(
            input_channels, w * 16, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(w * 16)
        self.layer1 = self._make_layer(block, w * 16, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, w * 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, w * 64, num_blocks[2], stride=2)
        print("the number of classes", self.num_classes)
        self.linear = nn.Linear(w * 64, self.num_classes)

        self.apply(_weights_init)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion

        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.avg_pool2d(out, out.size()[3])
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


###############################################################################################################################################


def resnet50():
    model = torchvision.models.resnet50(weights=None)
    return model.cuda()


def get_blocks(net):
    return nn.Sequential(
        nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool),
        *net.layer1,
        *net.layer2,
        *net.layer3,
        *net.layer4,
    )


In [ ]:
%%writefile loaders.py
import torch
from torchvision import datasets, transforms
import os
import yaml

def cifar_dataloader(batch_size, config, slurm_tmpdir = None):

    dataset = config["dataset"]
    dataset_root = config["dataset_root"][dataset]

    if dataset == "cifar10":
        print("Load CIFAR10 dataset")
        normalize = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        train_transform = transforms.Compose(
            [
                transforms.RandomHorizontalFlip(),
                transforms.RandomCrop(32, 4),
                transforms.ToTensor(),
                normalize,
            ]
        )
        test_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                normalize,
            ]
        )
        train_ds = datasets.CIFAR10(
            root=dataset_root,
            train=True,
            download=True,
            transform=train_transform,
        )
        test_ds = datasets.CIFAR10(
            root=dataset_root,
            train=False,
            download=True,
            transform=test_transform,
        )
    elif dataset == "cifar100":
        print("Load CIFAR100 dataset")
        normalize = transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762))
        train_transform = transforms.Compose(
            [
                transforms.RandomHorizontalFlip(),
                transforms.RandomCrop(32, 4),
                transforms.ToTensor(),
                normalize,
            ]
        )
        test_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                normalize,
            ]
        )
        train_ds = datasets.CIFAR100(
            root=dataset_root,
            train=True,
            download=True,
            transform=train_transform,
        )
        test_ds = datasets.CIFAR100(
            root=dataset_root,
            train=False,
            download=True,
            transform=test_transform,
        )

    elif dataset == "imagenet":
        print("Load ImageNet dataset")
        traindir = os.path.join(slurm_tmpdir, 'train')
        valdir = os.path.join(slurm_tmpdir, 'val')
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])

        train_ds = datasets.ImageFolder(
            traindir,
            transforms.Compose([
                transforms.RandomResizedCrop(224),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                normalize,
            ]))

        test_ds = datasets.ImageFolder(
            valdir,
            transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(224),
                transforms.ToTensor(),
                normalize,
            ]))

    elif dataset == "svhn":
        print("Load SVHN dataset")
        normalize = transforms.Normalize((0.4377, 0.4438, 0.4728), (0.1980, 0.2010, 0.1970))
        train_transform = transforms.Compose(
            [
                transforms.RandomCrop(32, padding=4),
                transforms.ToTensor(),
                normalize,
            ]
        )
        test_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                normalize,
            ]
        )
        train_ds = datasets.SVHN(
            root=dataset_root,
            split="train",
            download=False,
            transform=train_transform,
        )
        test_ds = datasets.SVHN(
            root=dataset_root,
            split="test",
            download=False,
            transform=test_transform,
        )
    elif dataset == "mnist":
        print("Load MNIST dataset")
        normalize = transforms.Normalize((0.1307,), (0.3081,))
        train_transform = transforms.Compose(
            [
                transforms.RandomCrop(28, padding=2),
                transforms.ToTensor(),
                normalize,
            ]
        )
        test_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                normalize,
            ]
        )
        train_ds = datasets.MNIST(
            root=dataset_root,
            train=True,
            download=True,
            transform=train_transform,
        )
        test_ds = datasets.MNIST(
            root=dataset_root,
            train=False,
            download=True,
            transform=test_transform,
        )
    else:
        raise ValueError(f"Unsupported dataset: {dataset}")

    num_workers = config.get("num_workers", 0)
    if os.name == "nt":
        num_workers = 0

    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=True,
        drop_last=True,
        pin_memory=True,
    )
    test_loader = torch.utils.data.DataLoader(
        test_ds,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=False,
        drop_last=False,
        pin_memory=True,
    )

    return train_loader, test_loader


In [ ]:
%%writefile run_logging.py
from __future__ import annotations

import json
import logging
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

_logger = logging.getLogger("sparse_rebasin")
_metrics_file: Path | None = None

def _json_safe(value: Any) -> Any:

    if isinstance(value, dict):
        return {str(key): _json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(item) for item in value]
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if hasattr(value, "item"):
        try:
            return _json_safe(value.item())
        except (TypeError, ValueError):
            pass
    return str(value)

def start_run(name: str, config: dict[str, Any], run_dir: str | Path) -> None:

    global _metrics_file

    output_dir = Path(run_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    _metrics_file = output_dir / f"{name}_metrics.jsonl"

    if not _logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
        _logger.addHandler(handler)
    _logger.setLevel(logging.INFO)
    _logger.info("Started run %s", name)

    log_metrics({"event": "run_started", "run_name": name, "config": config})

def log_metrics(metrics: dict[str, Any]) -> None:

    if _metrics_file is None:
        raise RuntimeError("A run must be started before logging metrics")

    record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        **metrics,
    }
    with _metrics_file.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(_json_safe(record)) + "\n")

def finish_run() -> None:

    if _metrics_file is not None:
        log_metrics({"event": "run_finished"})
        _logger.info("Finished run")


In [ ]:
%%writefile utils.py
import copy
import math
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
from loaders import cifar_dataloader
from models import *
from torch.cuda.amp import autocast
from torch.nn.utils.prune import custom_from_mask

def evaluate(model, dataloader, device):

    loss_fn = nn.CrossEntropyLoss()
    model.eval()
    correct = 0
    total = 0
    test_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)

            pred = outputs.argmax(dim=1)
            correct += (y == pred).sum().item()
            total += y.size(0)
            loss = loss_fn(outputs, y)
            test_loss += loss.item()

    test_accuracy = correct / total
    test_loss /= len(dataloader)
    return test_accuracy * 100, test_loss

def evaluate_top5(model, dataloader, device):

    loss_fn = nn.CrossEntropyLoss()
    model.eval()
    correct_top5 = 0
    total = 0
    test_loss = 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = loss_fn(outputs, y)
            test_loss += loss.item()

            _, pred_top5 = outputs.topk(5, dim=1)

            correct_top5 += sum([y[i] in pred_top5[i] for i in range(y.size(0))])
            total += y.size(0)

    test_accuracy_top5 = correct_top5 / total
    test_loss /= len(dataloader)

    return test_accuracy_top5 * 100, test_loss

def evaluate_merged_models(model_A_dense, permuted_model_b, alpha, config):
    device = config["device"]
    model_a = merge_models(model_A_dense, permuted_model_b, alpha, config)
    return model_a

def check_sparse_gradients(model):

    for name, param in model.named_parameters():
        if param.grad is not None:
            sparsity = float(torch.sum(param.grad == 0)) / torch.numel(param.grad)
            print(f"Gradient sparsity for {name}: {sparsity:.2f}")

def sparse_grad_tolerance(model, tolerance=0.0):

    for name, module in model.named_modules():
        if not hasattr(module, "weight_mask"):
            continue
        gradient = module.weight_orig.grad
        if gradient is None:
            continue
        masked = gradient[module.weight_mask == 0]
        if not torch.isfinite(masked).all() or (masked.abs() > tolerance).any():
            raise AssertionError(f"Nonzero or nonfinite masked gradients in {name}")

def calculate_mask_sparsity(weight_mask):
    total_params = torch.numel(weight_mask)
    zero_params = torch.sum(weight_mask == 0).item()
    sparsity = zero_params / total_params
    return sparsity, zero_params, total_params

def calculate_overall_sparsity_from_pth(model):
    total_zero_params = 0
    total_params = 0

    for name, param in model.state_dict().items():
        if 'weight_mask' in name:
            _, zero_params, num_params = calculate_mask_sparsity(param)
            total_zero_params += zero_params
            total_params += num_params

    overall_sparsity = total_zero_params / total_params if total_params > 0 else 0
    return overall_sparsity

def calculate_neuron_variances(model, unbiased=False):
    neuron_variances = {}
    with torch.no_grad():
        for name, param in model.named_parameters():
            if param.requires_grad and param.dim() > 1:
                variances = param.var(dim=list(range(1, param.dim())), unbiased=unbiased)
                neuron_variances[name] = variances
    return neuron_variances

def calculate_neuron_means(model):
    neuron_means = {}
    with torch.no_grad():
        for name, param in model.named_parameters():
            if param.requires_grad and param.dim() > 1:
                means = param.mean(dim=list(range(1, param.dim()))).detach()
                neuron_means[name] = means
    return neuron_means

def calculate_layer_variances(model, unbiased=False):
    variances = {}
    with torch.no_grad():
        for name, param in model.named_parameters():
            if param.requires_grad and param.dim() > 1:
                var = param.var(unbiased=unbiased).item()
                variances[name] = var
    return variances

def calculate_layer_means(model):
    means = {}
    with torch.no_grad():
        for name, param in model.named_parameters():
            if param.requires_grad and param.dim() > 1:
                mean = param.mean().item()
                means[name] = mean
    return means

def get_fan_in_tensor_mike(mask: torch.Tensor) -> torch.Tensor:
    if mask.dim() < 2:
        raise ValueError(
            "Fan in can not be computed for tensor with fewer than 2 dimensions"
        )
    if mask.dtype == torch.bool:
        fan_in_tensor = mask.sum(dim=list(range(1, mask.dim())))
    else:
        fan_in_tensor = (mask != 0.0).sum(dim=list(range(1, mask.dim())))
    return fan_in_tensor

def scale_model_with_fan_in(B_init_dense, A_sparse, config, mode: str = "fan_in", neuron_basis = True):
    if mode.lower() != "fan_in":
        raise NotImplementedError(
            "Only mode==`fan_in` has currently been implemented at this time."
        )

    if neuron_basis:
        layer_variances = calculate_neuron_variances(B_init_dense)
        layer_means = calculate_neuron_means(B_init_dense)

    else:
        layer_variances = calculate_layer_variances(B_init_dense)
        layer_means = calculate_layer_means(B_init_dense)

    print("print layer variance keys:", layer_variances.keys())
    print("print layer means keys:", layer_means.keys())

    for (name_B, m_B), (name_A, m_A) in zip(B_init_dense.named_modules(), A_sparse.named_modules()):
        if isinstance(m_B, (nn.Conv2d, nn.Linear)) and isinstance(m_A, (nn.Conv2d, nn.Linear)):
            with torch.no_grad():
                if config['model_type'] == 'ResNet':
                    if not hasattr(m_A, 'weight_mask'):
                        print(f"Skipping layer {name_A} ({m_A}) as it does not have weight_mask.")
                        continue
                mask = m_A.weight_mask
                if mask.shape != m_B.weight.shape:
                    raise ValueError(f"Sparsity mask and weight tensor shape do not match for layer {name_A}!")
                if 0 in m_B.weight.shape:
                    print(f"Initializing zero-element tensors is a no-op for layer {name_B}")
                    return m_B.weight
                fan_in = get_fan_in_tensor_mike(mask)

                weight_name = f"{name_B}.weight_orig"

                if weight_name not in layer_means or weight_name not in layer_variances:
                    print(f"Layer {name_B} not found in layer_means or layer_variances.")
                    continue

                mean_B = layer_means[weight_name]
                variance_B = layer_variances[weight_name]

                target_variance = 2 / fan_in
                if neuron_basis:
                    for i in range(m_B.weight.shape[0]):
                        if fan_in[i] != 0:

                            m_B.weight.data[i] -= mean_B[i]
                            sqrt_target_variance = math.sqrt(target_variance[i]) / math.sqrt(variance_B[i])
                            m_B.weight.data[i] *= sqrt_target_variance

                        elif fan_in[i] == 0:
                            m_B.weight.data[i] = 0
                else:
                    for i in range(m_B.weight.shape[0]):
                        if fan_in[i] != 0:

                            m_B.weight.data[i] -= mean_B
                            sqrt_target_variance = math.sqrt(target_variance[i]) / math.sqrt(variance_B)
                            m_B.weight.data[i] *= sqrt_target_variance

                        elif fan_in == 0:
                            m_B.weight.data[i] = 0

    return B_init_dense

def compute_loss_and_backward(model, data, target, loss_fn):
    model.train()
    output = model(data)
    loss = loss_fn(output, target)
    model.zero_grad()
    loss.backward()
    return loss

def calculate_gradient_norms(model):
    grad_norms = {}
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            if module.weight.grad is not None:
                grad_norms[f"{name}.weight"] = module.weight.grad.norm().item()
    return grad_norms

def compute_gradient_norms(scaled_model, train_dl, config):

    for data, target in train_dl:
        data, target = data.to(config["device"]), target.to(config["device"])
        break

    loss_fn = nn.CrossEntropyLoss()
    compute_loss_and_backward(scaled_model, data, target, loss_fn)
    grad_norms = calculate_gradient_norms(scaled_model)

    print("Gradient norms after scaling and resetting BN stats:", grad_norms)

def transfer_sparsity_adnan(model_A, model_B):

    modules_to_replace = {}

    for (name_A, module_A), (name_B, module_B) in zip(model_A.features.named_modules(), model_B.features.named_modules()):
        assert(type(module_A) is type(module_B) and name_A == name_B)

        if name_A != 'classifier' and hasattr(module_A, "weight_mask"):
            print('Replacing layer in model B with masked layer:', name_A)
            modules_to_replace[name_A] = custom_from_mask(copy.deepcopy(module_B), name="weight", mask=module_A.weight_mask)

        else:
            print('Skipping layer in model A when copying masks:', name_A)

    print(modules_to_replace)
    for module_name, module in modules_to_replace.items():
        with torch.no_grad():
            assert(hasattr(model_B.features, module_name))
            setattr(model_B.features, module_name, module)
            assert(hasattr(getattr(model_B.features, module_name), 'weight_mask'))

    modules_to_replace = {}
    for (name_A, module_A), (name_B, module_B) in zip(model_A.classifier.named_modules(), model_B.classifier.named_modules()):
        assert(type(module_A) is type(module_B) and name_A == name_B)

        if hasattr(module_A, "weight"):
            print('Replacing layer in model B with masked layer:', name_A)
            modules_to_replace[name_A] = custom_from_mask(copy.deepcopy(module_B), name="weight", mask=module_A.weight_mask)
        else:
            print('Skipping layer in model A when copying masks:', name_A)

    print(modules_to_replace)
    for module_name, module in modules_to_replace.items():
        with torch.no_grad():
            setattr(model_B, 'classifier', module)
            assert(hasattr(getattr(model_B, 'classifier'), 'weight_mask'))

def check_hooks(model):
    def check_module_hooks(module):
        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)) and hasattr(module, "weight_mask"):
            print(module, module._forward_pre_hooks)
            if len(module._forward_pre_hooks) == 0:
                print('Error')

        for child in module.children():
            check_module_hooks(child)

    check_module_hooks(model)

def get_model(config):
    if config["model_type"] == "VGG11":
        num_classes = dataset_num_classes.get(config['dataset'], 10)
        return VGG11_nofc(config, "VGG11", num_classes, init_weights=config["init_weights"]).to(config["device"])
    elif config["model_type"] == "ResNet":
        return ResNet(BasicBlock, [3, 3, 3], config, w=config["width_multiplier"]).to(config["device"])
    elif config["model_type"] == "ResNet50":
        return resnet50().to(config["device"])
    elif config["model_type"] == "LeNet":
        return LeNet(
            config,
            w=config.get("width_multiplier", 1),
            init_weights=config.get("init_weights", True),
        ).to(config["device"])
    else:
        raise ValueError(f"Unsupported model type: {config['model_type']}")

def merge_models(m0, m1, alpha, config):
    net = get_model(config)
    sd0, sd1 = m0.state_dict(), m1.state_dict()
    sd_alpha = {
        k: (1 - alpha) * sd0[k].to(config['device']) + alpha * sd1[k].to(config['device'])
        for k in sd0.keys()
        if k in sd1
    }
    net.load_state_dict(sd_alpha, strict=False)
    return net

"""
The following function is taken from REPAIR.
"""

def reset_bn_stats(model, loader):
    device = next(model.parameters()).device

    for m in model.modules():
        if type(m) == nn.BatchNorm2d:
            m.momentum = None
            m.reset_running_stats()

    model.train()
    with torch.no_grad(), autocast(enabled=device.type == "cuda"):
        for images, _ in loader:
            output = model(images.to(device))

def reset_bn_stats_sparse(model, loader, iter = 40, reset_params=False):

    for m in model.modules():
        if type(m) == nn.BatchNorm2d:
            m.momentum = 0.1
            m.reset_running_stats()
            if reset_params == True:
                m.reset_parameters()

    model.train()
    with torch.no_grad():
        for epoch in range(iter):
            for images, _ in loader:
                output = model(images.cuda())

def compare_model_weights(model1, model2):
    for (name1, param1), (name2, param2) in zip(model1.named_parameters(), model2.named_parameters()):
        if "bias" not in name1 and "bias" not in name2:
            if not torch.equal(param1, param2):
                print(f"Difference found in layer: {name1}")
                return False
    print("All weights match for all layers.")
    return True

def transfer_sparsity_resnet(model_A, model_B):

    modules_to_replace = {}

    for (name_A, module_A), (name_B, module_B) in zip(model_A.named_modules(), model_B.named_modules()):
        if type(module_A) is not type(module_B) or name_A != name_B:
            print(f"Mismatch found:")
            print(f"- model_A: {name_A}, type: {type(module_A)}")
            print(f"- model_B: {name_B}, type: {type(module_B)}")
        assert(type(module_A) is type(module_B) and name_A == name_B)

        if hasattr(module_A, "weight_mask"):
            print('Replacing layer in model B with masked layer:', name_A)
            modules_to_replace[name_A] = custom_from_mask(copy.deepcopy(module_B), name="weight", mask=module_A.weight_mask)
        else:
            print('Skipping layer in model A when copying masks:', name_A)

    for module_name, module in modules_to_replace.items():
        with torch.no_grad():
            mod_attr = model_B
            for i in range(len(module_name.split('.'))-1):
                if len(module_name.split('.')[i])!=1:
                    mod_attr = getattr(model_B, module_name.split('.')[i])
                else:
                    mod_attr = mod_attr[int(module_name.split('.')[i])]

            assert(hasattr(mod_attr, module_name.split('.')[-1]))
            setattr(mod_attr, module_name.split('.')[-1], module)
            assert(hasattr(getattr(mod_attr, module_name.split('.')[-1]), 'weight_mask'))

class TrackLayer(nn.Module):
    def __init__(self, layer, one_d=False):
        super().__init__()
        self.layer = layer
        dim = layer.conv3.out_channels
        self.bn = nn.BatchNorm2d(dim)

    def get_stats(self):
        return (self.bn.running_mean, self.bn.running_var.sqrt())

    def forward(self, x):
        x1 = self.layer(x)
        self.bn(x1)
        return x1

class ResetLayer(nn.Module):
    def __init__(self, layer, one_d=False):
        super().__init__()
        self.layer = layer
        dim = layer.conv3.out_channels
        self.bn = nn.BatchNorm2d(dim)

    def set_stats(self, goal_mean, goal_std):
        self.bn.bias.data = goal_mean
        self.bn.weight.data = goal_std

    def forward(self, x):
        x1 = self.layer(x)
        return self.bn(x1)

def make_tracked_net(net, config):
    net1 = get_model(config)
    net1.load_state_dict(net.state_dict())
    for i in range(4):
        layer = getattr(net1, 'layer%d' % (i+1))
        for j, block in enumerate(layer):
            layer[j] = TrackLayer(block).cuda()
    return net1

def make_repaired_net(net, config):
    net1 = get_model(config)
    net1.load_state_dict(net.state_dict())
    for i in range(4):
        layer = getattr(net1, 'layer%d' % (i+1))
        for j, block in enumerate(layer):
            layer[j] = ResetLayer(block).cuda()
    return net1

def repair_merged_model(permuted_model_0, model_1, model_a, train_dl, config, alpha=0.5):

    model0 = evaluate_merged_models(permuted_model_0, model_1, 0, config)
    model1 = evaluate_merged_models(permuted_model_0, model_1, 1, config)

    wrap0 = make_tracked_net(model0)
    wrap1 = make_tracked_net(model1)
    reset_bn_stats(wrap0, train_dl)
    reset_bn_stats(wrap1, train_dl)

    wrap_a = make_repaired_net(model_a)

    for track0, track1, reset_a in zip(wrap0.modules(), wrap1.modules(), wrap_a.modules()):
        if not isinstance(track0, TrackLayer):
            continue
        assert (isinstance(track0, TrackLayer)
                and isinstance(track1, TrackLayer)
                and isinstance(reset_a, ResetLayer))

        mu0, std0 = track0.get_stats()
        mu1, std1 = track1.get_stats()

        goal_mean = (1 - alpha) * mu0 + alpha * mu1
        goal_std = (1 - alpha) * std0 + alpha * std1
        reset_a.set_stats(goal_mean, goal_std)

    reset_bn_stats(wrap_a, train_dl)

    return wrap_a

def update_config_with_slurm_tmpdir(config, slurm_tmpdir):
    config['dataset_root']['imagenet'] = os.path.join(slurm_tmpdir, 'imagenet')
    return config


In [ ]:
%%writefile training.py
import torch
import torch.nn as nn
import torch.optim as optim
import math
from loaders import cifar_dataloader
from utils import evaluate, sparse_grad_tolerance
from run_logging import log_metrics
import yaml

def train_epoch(model, dataloader, optimizer, device, sparse=False):

    model.train()
    running_loss = 0
    loss_fn = nn.CrossEntropyLoss()

    for batch_index, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()

        pred = model(X)
        loss = loss_fn(pred, y)
        loss_value = loss.item()
        if not math.isfinite(loss_value):
            raise FloatingPointError("Nonfinite training loss; stop before producing a misleading final result")
        running_loss += loss_value

        loss.backward()

        if sparse and batch_index == 0:
            sparse_grad_tolerance(model)
        optimizer.step()

    return running_loss / len(dataloader)

def trainer(model, optimizer_config, epochs, batch_size, device, model_name, config, sparse=False, training_type="naive", k=0):

    comparison = config.get("comparison", {})
    save_rewind = comparison.get("enabled", False) and model_name in ("Model_A_Dense", "Model_B_Dense")
    rewind_epoch = int(comparison.get("rewind_epoch", 0))
    if save_rewind:
        if not 0 <= rewind_epoch <= epochs:
            raise ValueError(f"{model_name}: rewind epoch {rewind_epoch} exceeds dense training epochs {epochs}")
        if rewind_epoch == 0:
            torch.save(model.state_dict(), f"{config['save_path']}{model_name}_rewind_0_seed_{config['seed']}.pth")

    optimizer_type = getattr(optim, optimizer_config["type"])
    lr = optimizer_config["sparse_lr"] if sparse else optimizer_config["lr"]
    optimizer = optimizer_type(
        model.parameters(),
        lr=lr,
        momentum=optimizer_config["momentum"],
        weight_decay=optimizer_config["weight_decay"],
    )
    lrs = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs, eta_min=0.0)
    train_dl, test_dl = cifar_dataloader(batch_size, config)

    for epoch in range(epochs):
        train_loss_est = train_epoch(model, train_dl, optimizer, device, sparse=sparse)
        if save_rewind and epoch + 1 == rewind_epoch:
            torch.save(model.state_dict(), f"{config['save_path']}{model_name}_rewind_{rewind_epoch}_seed_{config['seed']}.pth")
        if not sparse and config.get("save_legacy_checkpoints", True) and epoch % 5 == 0:
            torch.save(model.state_dict(), config['save_path'] + model_name + '_sparsity_' + str(config['pruning']['sparsity'])+ '_seed_' + str(config['seed']) + '_epoch_' + str(epoch))
        test_accu, test_loss = evaluate(model, test_dl, device)
        print(
            f"Epoch: {epoch+1:2d} - Test accuracy: {test_accu:.2f} - Test loss: {test_loss:.4f} - ",
            f"Train loss est.: {train_loss_est:.4f} - Learning rate: {optimizer.param_groups[0]['lr']:.4f}",
        )
        log_metrics({
            "event": "epoch_summary", "model": model_name, "epoch": epoch + 1,
            "train_loss": train_loss_est, "test_loss": test_loss,
            "test_accuracy": test_accu, "learning_rate": optimizer.param_groups[0]["lr"],
        })
        if sparse:
            if training_type == "naive":
                log_metrics(
                    {
                        f"{model_name} Naive Epoch": epoch + 1,
                        f"{model_name} Naive Train Loss": train_loss_est,
                        f"{model_name} Naive Test Loss": test_loss,
                        f"{model_name} Naive Test Accuracy": test_accu,
                    }
                )
            elif training_type == "permuted":
                log_metrics(
                    {
                        f"{model_name}  Epoch": epoch + 1,
                        f"{model_name}  Train Loss": train_loss_est,
                        f"{model_name}  Test Loss": test_loss,
                        f"{model_name}  Test Accuracy": test_accu,
                    }
                )
            elif training_type == "LTH":
                log_metrics(
                    {
                        f"{model_name}  Epoch": epoch + 1,
                        f"{model_name}  Train Loss": train_loss_est,
                        f"{model_name}  Test Loss": test_loss,
                        f"{model_name}  Test Accuracy": test_accu,
                    }
                )
            elif training_type == "progressive":
                log_metrics(
                    {
                        f"{model_name} Progressive Epoch": epoch + 1,
                        f"{model_name} Progressive Train Loss": train_loss_est,
                        f"{model_name} Progressive Test Loss": test_loss,
                        f"{model_name} Progressive Test Accuracy": test_accu,
                    }
                )
        else:
            log_metrics(
                {
                    f"{model_name}  Epoch": epoch + 1,
                    f"{model_name}  Train Loss": train_loss_est,
                    f"{model_name}  Test Loss": test_loss,
                    f"{model_name}  Test Accuracy": test_accu,
                }
            )
        lrs.step()


In [ ]:
%%writefile progressive.py
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

def eligible_modules(model: nn.Module, model_type: str) -> list[tuple[str, nn.Module]]:

    excluded = ()
    if model_type == "ResNet":
        excluded = ("shortcut",)
    elif model_type == "ResNet50":
        excluded = ("shortcut", "downsample")
    elif model_type not in {"VGG11", "LeNet"}:
        raise ValueError(f"Unsupported model type: {model_type}")

    return [
        (name, module)
        for name, module in model.named_modules()
        if isinstance(module, (nn.Conv2d, nn.Linear))
        and not any(part in name for part in excluded)
    ]

def _mask(module: nn.Module) -> torch.Tensor:
    if not hasattr(module, "weight_mask"):
        raise ValueError("All progressive layers must have a weight_mask")
    return module.weight_mask.detach()

def mask_statistics(model: nn.Module, model_type: str) -> tuple[int, int]:

    active = total = 0
    for _, module in eligible_modules(model, model_type):
        mask = _mask(module)
        active += int(mask.count_nonzero().item())
        total += mask.numel()
    return active, total

def snapshot_masks(model: nn.Module, model_type: str) -> dict[str, torch.Tensor]:

    return {
        name: _mask(module).bool().clone()
        for name, module in eligible_modules(model, model_type)
    }

def mask_retention_statistics(
    model: nn.Module,
    model_type: str,
    initial_masks: dict[str, torch.Tensor],
    grown_masks: dict[str, torch.Tensor],
) -> dict[str, int | float]:

    final_masks = snapshot_masks(model, model_type)
    names = {name for name, _ in eligible_modules(model, model_type)}
    if set(initial_masks) != names or set(grown_masks) != names:
        raise ValueError("Mask snapshots do not match the model's eligible layers")

    initial = torch.cat([initial_masks[name].reshape(-1) for name in sorted(names)])
    grown = torch.cat([grown_masks[name].reshape(-1) for name in sorted(names)])
    final = torch.cat([final_masks[name].reshape(-1) for name in sorted(names)])
    added = grown & ~initial

    initial_count = int(initial.sum().item())
    added_count = int(added.sum().item())
    return {
        "initial_active": initial_count,
        "added_active": added_count,
        "final_active": int(final.sum().item()),
        "initial_retained": int((initial & final).sum().item()),
        "added_retained": int((added & final).sum().item()),
        "initial_retention": (
            float((initial & final).sum().item() / initial_count)
            if initial_count
            else 1.0
        ),
        "added_retention": (
            float((added & final).sum().item() / added_count) if added_count else 1.0
        ),
    }

def validate_mask_compatibility(
    source_model: nn.Module, target_model: nn.Module, model_type: str
) -> None:
    source_layers = eligible_modules(source_model, model_type)
    target_layers = eligible_modules(target_model, model_type)
    if [name for name, _ in source_layers] != [name for name, _ in target_layers]:
        raise ValueError("Source and target progressive layers do not match")
    for (name_a, source), (name_b, target) in zip(source_layers, target_layers):
        if source.weight.shape != target.weight.shape:
            raise ValueError(
                f"Layer shape mismatch for {name_a}/{name_b}: "
                f"{tuple(source.weight.shape)} != {tuple(target.weight.shape)}"
            )

def _install_masks(
    model: nn.Module, masks: dict[str, torch.Tensor], model_type: str
) -> None:

    for name, module in eligible_modules(model, model_type):
        if hasattr(module, "weight_mask"):
            module.weight_mask.data.copy_(
                masks[name].to(
                    device=module.weight_mask.device, dtype=module.weight_mask.dtype
                )
            )
        else:
            prune.custom_from_mask(
                module, "weight", masks[name].to(device=module.weight.device)
            )

def transfer_mask(
    source_model: nn.Module, target_model: nn.Module, model_type: str
) -> nn.Module:

    validate_mask_compatibility(source_model, target_model, model_type)
    masks = {
        name: _mask(source).to(dtype=target.weight.dtype).clone()
        for (name, source), (_, target) in zip(
            eligible_modules(source_model, model_type),
            eligible_modules(target_model, model_type),
        )
    }
    _install_masks(target_model, masks, model_type)
    return target_model

def grow_mask(
    model: nn.Module,
    model_type: str,
    target_density: float,
    selection: str = "magnitude",
    generator: torch.Generator | None = None,
) -> tuple[int, int]:

    if not 0.0 <= target_density <= 1.0:
        raise ValueError("target_density must be between 0 and 1")
    if selection not in {"magnitude", "random", "gradient"}:
        raise ValueError("selection must be 'magnitude', 'random', or 'gradient'")

    layers = eligible_modules(model, model_type)
    if selection == "gradient":
        for _, module in layers:
            weights = getattr(module, "weight_orig", module.weight)
            if weights.grad is None:
                raise ValueError(
                    "gradient selection requires gradients on all progressive layers"
                )

    old_active, total = mask_statistics(model, model_type)
    target_active = int(round(target_density * total))
    if target_active < old_active:
        raise ValueError(
            f"Progressive growth cannot reduce active weights: "
            f"{old_active} -> {target_active}"
        )
    add_count = target_active - old_active
    if add_count == 0:
        return old_active, total

    candidate_indices: list[torch.Tensor] = []
    candidate_scores: list[torch.Tensor] = []
    for name, module in layers:
        mask = _mask(module).bool()
        weights = getattr(module, "weight_orig", module.weight)
        if selection == "gradient":
            scores = weights.grad.detach().abs()
        else:
            scores = weights.detach().abs()
        inactive = ~mask.reshape(-1)
        candidate_indices.append(inactive.nonzero(as_tuple=False).flatten())
        candidate_scores.append(scores.reshape(-1)[inactive])

    candidate_count = sum(indices.numel() for indices in candidate_indices)
    if add_count > candidate_count:
        raise ValueError("Requested density exceeds available inactive weights")

    scores = torch.cat(candidate_scores)
    if selection == "magnitude":
        _, selected = torch.topk(scores, add_count, largest=True, sorted=False)
    elif selection == "random":
        order = torch.randperm(candidate_count, generator=generator)
        selected = order[:add_count].to(device=scores.device)
    else:
        _, selected = torch.topk(scores, add_count, largest=True, sorted=False)

    masks = {name: _mask(module).clone() for name, module in layers}
    offsets = torch.tensor(
        [
            0,
            *torch.cumsum(
                torch.tensor(
                    [indices.numel() for indices in candidate_indices],
                    dtype=torch.long,
                ),
                dim=0,
            ).tolist(),
        ],
        device=scores.device,
    )

    layer_indices = torch.bucketize(selected, offsets[1:-1], right=True)
    for layer_index, (name, _) in enumerate(layers):
        selected_in_layer = selected[layer_indices == layer_index]
        if selected_in_layer.numel() == 0:
            continue
        local_indices = selected_in_layer - offsets[layer_index]
        flat_mask = masks[name].reshape(-1)
        flat_mask[candidate_indices[layer_index][local_indices]] = 1
    _install_masks(model, masks, model_type)
    return mask_statistics(model, model_type)

def assert_density(
    model: nn.Module, model_type: str, expected_density: float, tolerance: float = 1e-6
) -> None:
    active, total = mask_statistics(model, model_type)
    actual = active / total

    tolerance = max(tolerance, 0.5 / total + 1e-12)
    if abs(actual - expected_density) > tolerance:
        raise AssertionError(
            f"Expected density {expected_density:.12f}, got {actual:.12f}"
        )


In [ ]:
%%writefile comparison.py
from __future__ import annotations

import hashlib
import json
import random
from pathlib import Path

import numpy as np
import torch

from loaders import cifar_dataloader
from progressive import eligible_modules, mask_statistics, transfer_mask
from run_logging import finish_run, log_metrics, start_run
from training import trainer
from utils import evaluate, get_model

METHOD_LABELS = {
    "naive": "LTH Problem (Naive)",
    "permuted": "Our Solution (Permuted)",
    "lth": "LTH Solution",
    "progressive": "Progressive (pruned + trained)",
}

def seed_training(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def initialize_dense_model(config, role):

    if role not in ("A", "B"):
        raise ValueError("Dense model role must be A or B")
    offset = 0 if role == "A" else 1_000_003
    seed_training(int(config["seed"]) + offset)
    return get_model(config)

def comparison_settings(config):
    settings = config.get("comparison", {})
    terminal_epoch = int(settings.get("total_epochs", config["sparse_epochs"]))
    rewind = int(settings.get("rewind_epoch", 0))
    minimum_post_prune = int(settings.get("min_post_prune_epochs", 1))
    if not 0 <= rewind < terminal_epoch:
        raise ValueError("Comparison requires 0 <= rewind_epoch < total_epochs (T)")
    if minimum_post_prune < 1:
        raise ValueError("min_post_prune_epochs must be at least 1")
    return terminal_epoch, rewind, minimum_post_prune

def rewind_path(config, role):
    _, rewind, _ = comparison_settings(config)
    return Path(config["save_path"]) / (
        f"Model_{role}_Dense_rewind_{rewind}_seed_{config['seed']}.pth"
    )

def load_rewind_model(config, role):
    path = rewind_path(config, role)
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing exact rewind checkpoint: {path}. Run pretraining with "
            "comparison.enabled=true. Legacy epoch_0 is AFTER one epoch "
            "and cannot be used as initialization (k=0)."
        )
    model = get_model(config)
    model.load_state_dict(torch.load(path, map_location=config["device"], weights_only=True))
    return model

def target_active_count(total, sparsity):
    return total - int(round(total * sparsity))

def pruning_rounds_required(active, total, sparsity, fraction):

    target = target_active_count(total, sparsity)
    if not 0 < fraction < 1 or active < target:
        raise ValueError("Invalid pruning fraction or a source mask below target density")
    rounds = 0
    while active > target:
        removed = min(int(round(fraction * active)), active - target)
        if removed <= 0:
            raise ValueError("Pruning fraction is too small to remove another weight")
        active -= removed
        rounds += 1
    return rounds

def growth_stage_epochs(config):

    growth = config["progressive"]
    targets = growth.get("target_densities", [])
    if not targets:
        raise ValueError("progressive.target_densities must contain at least one target")
    epochs = growth.get("epochs_per_stage", 1)
    if isinstance(epochs, int) and not isinstance(epochs, bool):
        result = [epochs] * len(targets)
    elif isinstance(epochs, (list, tuple)):
        result = list(epochs)
    else:
        raise ValueError("epochs_per_stage must be an integer or a list of integers")
    if len(result) != len(targets):
        raise ValueError("epochs_per_stage must have one entry per target density")
    if any(not isinstance(value, int) or isinstance(value, bool) or value < 1 for value in result):
        raise ValueError("Each growth stage must train for a positive integer number of epochs")
    return result

def progressive_budget(config, total_weights):
    terminal, rewind, minimum_post_prune = comparison_settings(config)
    growth = config["progressive"]
    growth_epochs = sum(growth_stage_epochs(config))
    rounds = pruning_rounds_required(
        int(round(growth["target_densities"][-1] * total_weights)),
        total_weights,
        config["pruning"]["sparsity"],
        config["pruning"]["prune_perc"],
    )
    if rounds > config["pruning"]["prune_epochs"]:
        raise ValueError(f"Progressive comparison needs at least {rounds} pruning rounds")
    pruning_epochs = rounds * int(growth.get(
        "train_epochs_per_prune", config["pruning"]["train_epochs_per_prune"]
    ))
    minimum_terminal = rewind + growth_epochs + pruning_epochs + minimum_post_prune
    if terminal < minimum_terminal:
        raise ValueError(
            f"T={terminal} is too small. Use T >= {minimum_terminal}: "
            f"k={rewind}, growth={growth_epochs}, pruning={pruning_epochs}, "
            f"post-pruning >= {minimum_post_prune}."
        )
    return growth_epochs, pruning_epochs

def configuration_signature(config):

    settings = {key: value for key, value in config.items() if key != "save_path"}
    return hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()

def record_initial_diagnostics(method, model, config):

    layers = []
    for name, module in eligible_modules(model, config["model_type"]):
        weights = module.weight_orig.detach()
        mask = module.weight_mask.detach()
        effective = weights * mask
        layers.append({
            "layer": name, "active_weights": int(mask.count_nonzero().item()),
            "total_weights": mask.numel(),
            "density": float(mask.count_nonzero().item() / mask.numel()),
            "dense_weight_rms": float(weights.square().mean().sqrt().item()),
            "masked_weight_rms": float(effective.square().mean().sqrt().item()),
        })
    _, rewind, _ = comparison_settings(config)
    record = {
        "event": "initial_diagnostics", "method": method,
        "completed_dense_epochs_at_rewind": rewind,
        "legacy_epoch_label_for_same_checkpoint": rewind - 1 if rewind > 0 else None,
        "initialization_checkpoint": str(rewind_path(config, "A" if method == "lth" else "B")),
        "initialization_seed": int(config["seed"]) + (0 if method == "lth" else 1_000_003),
        "layers": layers,
    }
    output = Path(config["save_path"]) / "diagnostics"
    output.mkdir(parents=True, exist_ok=True)
    (output / f"{method}_initial_seed_{config['seed']}.json").write_text(
        json.dumps(record, indent=2), encoding="utf-8"
    )
    log_metrics(record)
    return record

def record_final_result(method, model, config, stages, test_loader=None):

    terminal, rewind, minimum_post_prune = comparison_settings(config)
    if rewind + sum(stages.values()) != terminal:
        raise ValueError(f"{method}: completed epochs do not reach T={terminal}")
    if method == "progressive" and stages.get("post_pruning", 0) < minimum_post_prune:
        raise ValueError("Progressive result requires training after the last pruning step")
    active, total = mask_statistics(model, config["model_type"])
    expected = target_active_count(total, config["pruning"]["sparsity"])
    if active != expected:
        raise ValueError(f"{method}: final mask has {active} active weights, expected {expected}")
    if test_loader is None:
        _, test_loader = cifar_dataloader(config["batch_size"], config)
    accuracy, loss = evaluate(model, test_loader, config["device"])
    output = Path(config["save_path"]) / "final_results"
    output.mkdir(parents=True, exist_ok=True)
    checkpoint = output / f"{method}_seed_{config['seed']}.pth"
    torch.save(model.state_dict(), checkpoint)
    record = {
        "method": method,
        "label": METHOD_LABELS[method],
        "seed": config["seed"],
        "terminal_epoch": terminal,
        "rewind_epoch": rewind,
        "checkpoint_convention": "completed_dense_epochs",
        "legacy_epoch_label_for_same_checkpoint": rewind - 1 if rewind > 0 else None,
        "trained_epochs": sum(stages.values()),
        "stage_epochs": stages,
        "test_accuracy": float(accuracy),
        "test_loss": float(loss),
        "target_sparsity": config["pruning"]["sparsity"],
        "actual_sparsity": 1 - active / total,
        "active_weights": active,
        "total_weights": total,
        "initialization_checkpoint": str(rewind_path(config, "A" if method == "lth" else "B")),
        "checkpoint": str(checkpoint),
        "configuration_signature": configuration_signature(config),
    }
    result_path = output / f"{method}_seed_{config['seed']}.json"
    temporary = result_path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(record, indent=2), encoding="utf-8")
    temporary.replace(result_path)
    log_metrics({"event": "final_result", **record})
    print(f"FINAL {METHOD_LABELS[method]}: t=T={terminal}, accuracy={accuracy:.2f}%, loss={loss:.4f}")
    return record

def train_sparse_baselines(seed, config):

    terminal, rewind, _ = comparison_settings(config)
    output = Path(config["save_path"])
    suffix = f"sparsity_{config['pruning']['sparsity']}_seed_{seed}"
    source_path = output / f"model_a_sparse_{suffix}"
    permuted_path = output / f"permuted_model_a_sparse_{suffix}"

    for path in (source_path, permuted_path, rewind_path(config, "A"), rewind_path(config, "B")):
        if not path.is_file():
            raise FileNotFoundError(f"Missing checkpoint {path}; rerun comparison-enabled pretraining")
    records = []
    for method, role, mask_path, training_type in (
        ("naive", "B", source_path, "naive"),
        ("permuted", "B", permuted_path, "permuted"),
        ("lth", "A", source_path, "LTH"),
    ):
        model = load_rewind_model(config, role)
        source = torch.load(mask_path, map_location=config["device"], weights_only=False)
        transfer_mask(source, model, config["model_type"])
        del source
        active, total = mask_statistics(model, config["model_type"])
        if active != target_active_count(total, config["pruning"]["sparsity"]):
            raise ValueError(f"{method}: source mask has incorrect sparsity; rerun pretraining")

        seed_training(seed)
        start_run(f"{method}_{config['model_type']}_seed_{seed}", {
            "method": method, "T": terminal, "k": rewind,
        }, config["save_path"])
        record_initial_diagnostics(method, model, config)
        trainer(
            model, config["optimizer"], terminal - rewind, config["batch_size"],
            config["device"], f"Comparison_{method}", config,
            sparse=True, training_type=training_type,
        )
        records.append(record_final_result(method, model, config, {"sparse_training": terminal - rewind}))
        finish_run()
        del model
    return records


In [ ]:
%%writefile ablation_experiment.py
from __future__ import annotations

import copy
import csv
import hashlib
import io
import json
import math
from pathlib import Path
import random
import time
import zipfile

import numpy as np
import torch
from torch.nn.utils import prune
from torch.utils.data import DataLoader
import yaml

from comparison import comparison_settings, configuration_signature, growth_stage_epochs, seed_training, target_active_count
from loaders import cifar_dataloader
from progressive import eligible_modules, grow_mask, mask_statistics, snapshot_masks
from training import train_epoch
from utils import evaluate, get_model

CONDITIONS = ("fixed_phase_matched", "progressive")
LABELS = {"fixed_phase_matched": "Fixed mask\nmatched phases", "progressive": "Growth +\nre-pruning"}

def _member(names, suffix, required=True):
    matches = [name for name in names if name == suffix or name.endswith("/" + suffix)]
    if len(matches) != 1:
        if not matches and not required:
            return None
        raise ValueError(f"ZIP must contain exactly one {suffix!r}; found {len(matches)}")
    return matches[0]

def phase_plan(config, total_weights):

    terminal, rewind, minimum_post = comparison_settings(config)
    if config["optimizer"]["type"] != "SGD":
        raise ValueError("This ablation matches the reference SGD implementation; other optimizers need a separate protocol")
    if config["progressive"].get("selection", "magnitude") not in ("magnitude", "random"):
        raise ValueError("Only magnitude/random growth is supported; the legacy masked-gradient rule is not an informative gradient control")
    sparsity = float(config["pruning"]["sparsity"])
    fraction = float(config["pruning"]["prune_perc"])
    if not 0 < sparsity < 1 or not 0 < fraction < 1:
        raise ValueError("sparsity and prune_perc must lie strictly between zero and one")
    target = target_active_count(total_weights, sparsity)
    active = target
    optimizer = config["optimizer"]
    growth_optimizer = {"lr": float(optimizer["sparse_lr"]), "momentum": float(optimizer["momentum"]),
                        "weight_decay": float(optimizer["weight_decay"])}
    pruning_optimizer = {"lr": float(config["pruning"]["prune_lr"]), **optimizer.get("params", {})}
    plan = []
    for index, (density, epochs) in enumerate(zip(config["progressive"]["target_densities"], growth_stage_epochs(config))):
        next_active = round(float(density) * total_weights)
        if not active < next_active <= total_weights:
            raise ValueError("Growth densities must increase the active count and not exceed one")
        plan.append({"kind": "growth", "name": f"growth_{index + 1}", "epochs": epochs,
                     "active_weights": next_active, "optimizer": growth_optimizer.copy()})
        active = next_active
    q = config["progressive"].get("train_epochs_per_prune", config["pruning"]["train_epochs_per_prune"])
    if isinstance(q, bool) or not isinstance(q, int) or q < 1:
        raise ValueError("Pruning epochs per round must be a positive integer")
    rounds = 0
    while active > target:
        removed = min(round(active * fraction), active - target)
        if removed <= 0:
            raise ValueError("Pruning fraction is too small to remove another weight")
        active -= removed
        rounds += 1
        plan.append({"kind": "pruning", "name": f"pruning_{rounds}", "epochs": q,
                     "active_weights": active, "optimizer": pruning_optimizer.copy()})
    if rounds > config["pruning"]["prune_epochs"]:
        raise ValueError("Configured maximum pruning rounds is insufficient")
    post = terminal - rewind - sum(phase["epochs"] for phase in plan)
    if post < minimum_post:
        raise ValueError("The imported T-k budget leaves insufficient post-pruning training")
    post_optimizer = growth_optimizer.copy()
    post_optimizer["lr"] = float(config.get("comparison", {}).get("post_prune_lr", config["pruning"]["prune_lr"]))
    plan.append({"kind": "post_pruning", "name": "post_pruning", "epochs": post,
                 "active_weights": target, "optimizer": post_optimizer})
    return plan

def load_bundle(zip_path):

    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError("ZIP has duplicate member names")
        config_name = _member(names, "colab_config.yaml")
        config_bytes = archive.read(config_name)
        config = yaml.safe_load(config_bytes)
        if not isinstance(config, dict):
            raise ValueError("colab_config.yaml must contain a configuration mapping")
        if config.get("model_type") not in ("ResNet", "VGG11", "LeNet"):
            raise ValueError("Supported architectures: ResNet20 (ResNet), VGG11 and LeNet, with the bundled model definitions")
        if config.get("dataset") not in ("cifar10", "cifar100", "mnist"):
            raise ValueError("Automatic dataset loading supports CIFAR-10, CIFAR-100 and MNIST")
        if not config.get("comparison", {}).get("enabled"):
            raise ValueError("ZIP must come from a comparison-enabled run with exact rewind checkpoints")
        terminal, rewind, _ = comparison_settings(config)
        seed = int(config["seed"])
        b_name = _member(names, f"Model_B_Dense_rewind_{rewind}_seed_{seed}.pth")
        mask_name = _member(names, f"final_results/naive_seed_{seed}.pth", required=False)
        if mask_name is None:
            mask_name = _member(names, f"final_results/lth_seed_{seed}.pth")
        b_bytes, mask_bytes = archive.read(b_name), archive.read(mask_name)
        b_state = torch.load(io.BytesIO(b_bytes), map_location="cpu", weights_only=True)
        mask_state = torch.load(io.BytesIO(mask_bytes), map_location="cpu", weights_only=True)
        if not isinstance(b_state, dict) or not isinstance(mask_state, dict):
            raise ValueError("Expected tensor state_dict checkpoints")
        references_name = _member(names, "final_comparison.json", required=False)
        references = json.loads(archive.read(references_name)) if references_name else []
        derived_name = _member(names, "input_bundle_provenance.json", required=False)
        derived_from = json.loads(archive.read(derived_name)) if derived_name else None
        model_source = _member(names, "source/models.py", required=False)
        if model_source:
            archived_models = archive.read(model_source).decode("utf-8").replace("\r\n", "\n").strip()
            bundled_models = (Path(__file__).parent / "models.py").read_text(encoding="utf-8").replace("\r\n", "\n").strip()
            if archived_models != bundled_models:
                raise ValueError("ZIP uses different model definitions. Update the notebook's models.py before using a custom architecture; archived Python is not executed automatically")
    cpu_config = {**config, "device": "cpu"}
    model = get_model(cpu_config)

    model.load_state_dict(b_state, strict=True)
    masks = {}
    for name, module in eligible_modules(model, config["model_type"]):
        key = f"{name}.weight_mask"
        if key not in mask_state:
            raise ValueError(f"Source mask missing {key}")
        mask = mask_state[key].detach().cpu()
        if mask.shape != module.weight.shape or not bool(((mask == 0) | (mask == 1)).all()):
            raise ValueError(f"Invalid binary source mask or shape for {key}")
        masks[name] = mask.to(dtype=torch.bool).clone()
    if set(key for key in mask_state if key.endswith("weight_mask")) != {f"{name}.weight_mask" for name in masks}:
        raise ValueError("Archived mask layers differ from this implementation's eligible layers")
    total = sum(mask.numel() for mask in masks.values())
    active = sum(int(mask.count_nonzero()) for mask in masks.values())
    if active != target_active_count(total, config["pruning"]["sparsity"]):
        raise ValueError("Source mask has a different final sparsity than the configuration")
    for record in references:
        if (record["seed"] != seed or record["terminal_epoch"] != terminal or record["rewind_epoch"] != rewind
                or record["active_weights"] != active or record["total_weights"] != total
                or record.get("checkpoint_convention") != "completed_dense_epochs"
                or ("configuration_signature" in record and record["configuration_signature"] != configuration_signature(config))):
            raise ValueError("Historical results do not match the imported configuration/checkpoint convention")
    fingerprint = hashlib.sha256(config_bytes + b_bytes + mask_bytes).hexdigest()
    return {"config": config, "b_state": b_state, "masks": masks, "total_weights": total,
            "initial_active": active, "references": references, "plan": phase_plan(config, total),
            "provenance": {"zip_name": zip_path.name, "config_member": config_name,
                           "b_checkpoint_member": b_name, "source_mask_member": mask_name,
                           "bundle_fingerprint": fingerprint,
                           "derived_from": derived_from,
                           "mask_only_transfer": True, "checkpoint_convention": "completed_dense_epochs"}}

def _state_digest(state):
    digest = hashlib.sha256()
    for name, value in sorted(state.items()):
        digest.update(name.encode())
        digest.update(str((value.dtype, tuple(value.shape))).encode())
        digest.update(value.detach().cpu().reshape(-1).view(torch.uint8).numpy().tobytes())
    return digest.hexdigest()

def starting_model(bundle, config):
    model = get_model(config)
    model.load_state_dict(bundle["b_state"], strict=True)
    for name, module in eligible_modules(model, config["model_type"]):
        prune.custom_from_mask(module, "weight", bundle["masks"][name].to(device=config["device"], dtype=module.weight.dtype))
    return model

def _seed_worker(worker_id):
    seed = torch.initial_seed() % (2**32)
    np.random.seed(seed)
    random.seed(seed)

def phase_loader(base, seed):

    return DataLoader(base.dataset, batch_size=base.batch_size, shuffle=True,
                      num_workers=base.num_workers, pin_memory=base.pin_memory,
                      drop_last=base.drop_last, worker_init_fn=_seed_worker,
                      generator=torch.Generator().manual_seed(seed))

def run_condition(bundle, config, condition, training_seed, output_dir, train_loader, test_loader):
    if condition not in CONDITIONS:
        raise ValueError(f"Unknown condition: {condition}")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)
    seed_training(training_seed)
    model = starting_model(bundle, config)
    initial_digest = _state_digest(model.state_dict())
    initial_masks = snapshot_masks(model, config["model_type"])
    growth_generator = torch.Generator().manual_seed(int(config["progressive"].get("growth_seed", config["seed"])))
    started = time.perf_counter()
    terminal, rewind, _ = comparison_settings(config)
    completed = updates = 0
    history = []
    with (output_dir / "epochs.jsonl").open("x", encoding="utf-8") as log:
        for phase_index, phase in enumerate(bundle["plan"]):
            if condition == "progressive":
                if phase["kind"] == "growth":
                    grow_mask(model, config["model_type"], phase["active_weights"] / bundle["total_weights"],
                              selection=config["progressive"].get("selection", "magnitude"), generator=growth_generator)
                elif phase["kind"] == "pruning":
                    active, _ = mask_statistics(model, config["model_type"])
                    prune.global_unstructured([(module, "weight") for _, module in eligible_modules(model, config["model_type"])],
                                              pruning_method=prune.L1Unstructured,
                                              amount=active - phase["active_weights"])
            active, total = mask_statistics(model, config["model_type"])
            expected = phase["active_weights"] if condition == "progressive" else bundle["initial_active"]
            if active != expected:
                raise AssertionError("Unexpected mask size at phase boundary")

            data_seed = (int(training_seed) + 1_000_003 * (phase_index + 1)) % (2**32)
            seed_training(data_seed)
            loader = phase_loader(train_loader, data_seed)
            optimizer = torch.optim.SGD(model.parameters(), **phase["optimizer"])
            schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, phase["epochs"], eta_min=0.0)
            for epoch in range(phase["epochs"]):
                lr = optimizer.param_groups[0]["lr"]
                train_loss = train_epoch(model, loader, optimizer, config["device"], sparse=True)
                accuracy, test_loss = evaluate(model, test_loader, config["device"])
                if not all(math.isfinite(value) for value in (accuracy, test_loss, train_loss)):
                    raise FloatingPointError("Nonfinite measurement")
                completed += 1
                updates += len(loader)
                record = {"condition": condition, "training_seed": training_seed, "phase": phase["name"],
                          "phase_epoch": epoch + 1, "completed_epochs": completed, "absolute_epoch": rewind + completed,
                          "phase_data_seed": data_seed, "optimizer": phase["optimizer"], "learning_rate": lr,
                          "active_weights": active, "total_weights": total, "density": active / total,
                          "train_loss": train_loss, "test_loss": test_loss, "test_accuracy": accuracy}
                history.append(record)
                log.write(json.dumps(record) + "\n")
                log.flush()
                print(f"{condition} seed={training_seed} {phase['name']} {epoch + 1}/{phase['epochs']} "
                      f"t={rewind + completed}/{terminal} accuracy={accuracy:.2f}% lr={lr:.6g}", flush=True)
                schedule.step()
    final_masks = snapshot_masks(model, config["model_type"])
    if condition == "fixed_phase_matched" and any(not torch.equal(final_masks[name].cpu(), mask.cpu()) for name, mask in initial_masks.items()):
        raise AssertionError("Fixed control changed its mask")
    active, total = mask_statistics(model, config["model_type"])
    if completed != terminal - rewind or active != bundle["initial_active"]:
        raise AssertionError("Final epoch count or mask size differs between conditions")
    retained = sum(int((final_masks[name].bool() & mask.bool()).sum()) for name, mask in initial_masks.items())
    result = {"condition": condition, "training_seed": training_seed, "source_seed": config["seed"],
              "terminal_epoch": terminal, "rewind_epoch": rewind, "trained_epochs": completed, "optimizer_updates": updates,
              "test_accuracy": history[-1]["test_accuracy"], "test_loss": history[-1]["test_loss"],
              "train_loss": history[-1]["train_loss"], "active_weights": active, "total_weights": total,
              "initial_retention": retained / bundle["initial_active"], "initial_state_digest": initial_digest,
              "bundle_fingerprint": bundle["provenance"]["bundle_fingerprint"], "seconds": time.perf_counter() - started,
              "checkpoint": str(output_dir / "final_state_dict.pth")}
    with (output_dir / "final_state_dict.pth").open("xb") as stream:
        torch.save(model.state_dict(), stream)
    (output_dir / "result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result, history

def run_ablation(bundle, output_dir, training_seeds=(0, 1, 2), device="cuda", dataset_root=None, num_workers=2):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    if not training_seeds or len(set(training_seeds)) != len(training_seeds) or any(not isinstance(s, int) or s < 0 for s in training_seeds):
        raise ValueError("Provide distinct nonnegative integer training seeds")
    if any(output_dir.iterdir()):
        raise ValueError("Choose a new empty output directory; existing runs are never overwritten")
    config = copy.deepcopy(bundle["config"])
    config.update(device=device, num_workers=num_workers)
    config["dataset_root"] = {config["dataset"]: str(Path(dataset_root or output_dir.parent / "datasets") / config["dataset"])}
    (output_dir / "source_config.yaml").write_text(yaml.safe_dump(bundle["config"], sort_keys=False), encoding="utf-8")
    (output_dir / "ablation_config.yaml").write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    manifest = {**bundle["provenance"], "training_seeds": list(training_seeds), "conditions": list(CONDITIONS),
                "phase_plan": bundle["plan"], "fixed_source_mask_and_B_checkpoint": True,
                "paired_phase_data_seeds": True, "exact_resume": False,
                "interpretation": "Conditional training-seed variation for one source mask/B checkpoint; equal updates, not a FLOP match"}
    (output_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    (output_dir / "historical_results.json").write_text(json.dumps(bundle["references"], indent=2), encoding="utf-8")
    with (output_dir / "input_B_rewind_state_dict.pth").open("xb") as stream:
        torch.save(bundle["b_state"], stream)
    with (output_dir / "input_source_masks.pth").open("xb") as stream:
        torch.save(bundle["masks"], stream)
    train_loader, test_loader = cifar_dataloader(config["batch_size"], config)
    results = []
    for seed in training_seeds:

        order = CONDITIONS if training_seeds.index(seed) % 2 == 0 else CONDITIONS[::-1]
        pair = {}
        for condition in order:
            result, history = run_condition(bundle, config, condition, seed,
                                            output_dir / f"training_seed_{seed}" / condition, train_loader, test_loader)
            results.append(result)
            pair[condition] = (result, history)
            write_result_table(results, output_dir / "ablation_results.csv")
        a, b = [pair[condition] for condition in CONDITIONS]
        schedule_keys = ("absolute_epoch", "phase", "phase_epoch", "learning_rate", "optimizer", "phase_data_seed")
        if (a[0]["initial_state_digest"] != b[0]["initial_state_digest"]
                or a[0]["optimizer_updates"] != b[0]["optimizer_updates"]
                or [[row[key] for key in schedule_keys] for row in a[1]] != [[row[key] for key in schedule_keys] for row in b[1]]):
            raise AssertionError("Paired initial state, update count or optimizer/data schedule differs")
    (output_dir / "paired_checks.json").write_text(json.dumps({"passed": True, "training_seeds": list(training_seeds),
        "initial_state_optimizer_schedule_data_seeds_and_update_count_matched": True}, indent=2), encoding="utf-8")
    return results

def write_result_table(results, destination):
    with Path(destination).open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(results[0]))
        writer.writeheader()
        writer.writerows(results)

def make_plots(output_dir):

    import matplotlib.pyplot as plt

    output_dir = Path(output_dir)
    if not json.loads((output_dir / "paired_checks.json").read_text())["passed"]:
        raise ValueError("Paired protocol checks have not passed")
    results = [json.loads(path.read_text()) for path in sorted(output_dir.glob("training_seed_*/*/result.json"))]
    seeds = sorted({result["training_seed"] for result in results})
    lookup = {(result["condition"], result["training_seed"]): result for result in results}
    if len(results) != 2 * len(seeds) or any((condition, seed) not in lookup for condition in CONDITIONS for seed in seeds):
        raise ValueError("Missing or duplicate completed pairs")
    config = yaml.safe_load((output_dir / "ablation_config.yaml").read_text())
    model_label = "ResNet20" if config["model_type"] == "ResNet" else config["model_type"]
    title = f"{model_label} ×{config.get('width_multiplier', 1)} / {config['dataset']}"
    n = len(seeds)
    values = np.array([[lookup[condition, seed]["test_accuracy"] for seed in seeds] for condition in CONDITIONS])
    mean = values.mean(axis=1)
    sd = values.std(axis=1, ddof=1) if n > 1 else np.zeros(2)
    delta = values[1] - values[0]
    summary = {"n_training_seeds": n, "training_seeds": seeds,
               "fixed_source_mask_and_B_checkpoint": True,
               "conditions": {condition: {"mean_accuracy": float(mean[i]), "sample_sd": float(sd[i]) if n > 1 else None}
                              for i, condition in enumerate(CONDITIONS)},
               "paired_difference_pp": {"values": delta.tolist(), "mean": float(delta.mean()),
                                         "sample_sd": float(delta.std(ddof=1)) if n > 1 else None},
               "scope": "Training randomness conditional on one imported source mask and B checkpoint; no independent-initialization error bars"}
    (output_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    with (output_dir / "paired_differences.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(["training_seed", "fixed_accuracy", "progressive_accuracy", "progressive_minus_fixed_pp"])
        writer.writerows(zip(seeds, values[0], values[1], delta))

    fig, axis = plt.subplots(figsize=(3.4, 2.25), layout="constrained")
    for i, seed in enumerate(seeds):
        offset = (i - (n - 1) / 2) * 0.035
        axis.plot([offset, 1 + offset], values[:, i], color="0.65", marker="o", linewidth=0.8, markersize=3, alpha=0.8)
    axis.errorbar([0, 1], mean, yerr=sd if n > 1 else None, fmt="D", color="black", capsize=4,
                  markersize=4, label="Mean ± sample SD" if n > 1 else "Single training seed")
    axis.set_xticks([0, 1], [LABELS[condition] for condition in CONDITIONS], fontsize=8)
    axis.set_ylabel("Final test accuracy (%)", fontsize=8)
    axis.set_title(f"{title}\npaired training seeds n={n}; Δ={delta.mean():+.2f} pp", fontsize=8)
    axis.tick_params(labelsize=8)
    axis.set_xlim(-0.3, 1.3)
    axis.grid(axis="y", alpha=0.25)
    axis.legend(fontsize=6, loc="best")
    for ext in ("png", "pdf", "svg"):
        fig.savefig(output_dir / f"ablation_report.{ext}", dpi=300)
    plt.show()
    references = json.loads((output_dir / "historical_results.json").read_text())
    if references:
        order = {method: i for i, method in enumerate(("naive", "permuted", "lth", "progressive"))}
        references.sort(key=lambda row: order.get(row["method"], 99))
        old_fig, old_axis = plt.subplots(figsize=(6.4, 3.5), layout="constrained")
        bars = old_axis.bar([row["label"] for row in references], [row["test_accuracy"] for row in references])
        old_axis.bar_label(bars, fmt="%.2f%%", padding=3)
        old_axis.set_ylim(0, 100)
        old_axis.set_ylabel("Final test accuracy (%)")
        old_axis.set_title(f"Imported reference run — {title}\nsource seed={config['seed']}, k={config['comparison']['rewind_epoch']}, T={config['comparison']['total_epochs']}; not paired repetitions")
        old_axis.tick_params(axis="x", labelsize=7)
        for ext in ("png", "pdf"):
            old_fig.savefig(output_dir / f"historical_reference.{ext}", dpi=200)
        plt.show()
    histories = []
    for path in sorted(output_dir.glob("training_seed_*/*/epochs.jsonl")):
        histories.extend(json.loads(line) for line in path.read_text().splitlines())
    write_result_table(histories, output_dir / "all_learning_curves.csv")
    curve_fig, curve_axis = plt.subplots(figsize=(6.4, 3.6), layout="constrained")
    for condition in CONDITIONS:
        runs = [[row for row in histories if row["condition"] == condition and row["training_seed"] == seed] for seed in seeds]
        ys = np.array([[row["test_accuracy"] for row in run] for run in runs])
        xs = [row["absolute_epoch"] for row in runs[0]]
        line, = curve_axis.plot(xs, ys.mean(axis=0), label=LABELS[condition].replace("\n", " "))
        if n > 1:
            spread = ys.std(axis=0, ddof=1)
            curve_axis.fill_between(xs, ys.mean(axis=0) - spread, ys.mean(axis=0) + spread, alpha=0.15, color=line.get_color())
    curve_axis.set(xlabel="Completed epoch t", ylabel="Test accuracy (%)", title=f"Paired learning curves — {title}")
    curve_axis.legend()
    curve_axis.grid(alpha=0.25)
    curve_fig.savefig(output_dir / "ablation_learning_curves.png", dpi=200)
    plt.show()
    print(json.dumps(summary, indent=2))
    return summary

def export_results(output_dir, export_root):

    from datetime import datetime, timezone
    output_dir, export_root = Path(output_dir), Path(export_root)
    if export_root.resolve() == output_dir.resolve() or output_dir.resolve() in export_root.resolve().parents:
        raise ValueError("Export ZIP must be outside the directory being archived")
    export_root.mkdir(parents=True, exist_ok=True)
    stem = f"{output_dir.name}_results_{datetime.now(timezone.utc):%Y%m%d_%H%M%S_%f_UTC}"
    counter = 0
    while True:
        path = export_root / f"{stem}{'_' + str(counter) if counter else ''}.zip"
        try:
            stream = path.open("xb")
            break
        except FileExistsError:
            counter += 1
    with stream, zipfile.ZipFile(stream, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for file in sorted(output_dir.rglob("*")):
            if file.is_file():
                archive.write(file, file.relative_to(output_dir))
    return path

def copy_export(source, destination_dir):
    import shutil
    source, destination_dir = Path(source), Path(destination_dir)
    destination_dir.mkdir(parents=True, exist_ok=True)
    counter = 0
    while True:
        path = destination_dir / f"{source.stem}{'_' + str(counter) if counter else ''}.zip"
        try:
            target = path.open("xb")
            break
        except FileExistsError:
            counter += 1
    with target, source.open("rb") as stream:
        shutil.copyfileobj(stream, target)
    return path

def make_input_bundle(source_zip, destination):

    bundle = load_bundle(source_zip)
    config = bundle["config"]
    seed = config["seed"]
    rewind = config["comparison"]["rewind_epoch"]
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(destination, "x", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("colab_config.yaml", yaml.safe_dump(config, sort_keys=False))
        archive.writestr("final_comparison.json", json.dumps(bundle["references"], indent=2))
        archive.writestr("source/models.py", (Path(__file__).parent / "models.py").read_text(encoding="utf-8"))
        for name, state in [(f"Model_B_Dense_rewind_{rewind}_seed_{seed}.pth", bundle["b_state"]),
                            (f"final_results/naive_seed_{seed}.pth", {f"{name}.weight_mask": mask for name, mask in bundle["masks"].items()})]:
            buffer = io.BytesIO()
            torch.save(state, buffer)
            archive.writestr(name, buffer.getvalue())
        archive.writestr("input_bundle_provenance.json", json.dumps({
            "original_zip": Path(source_zip).name,
            "original_bundle_fingerprint": bundle["provenance"]["bundle_fingerprint"],
            "original_b_checkpoint": bundle["provenance"]["b_checkpoint_member"],
            "original_mask_checkpoint": bundle["provenance"]["source_mask_member"],
            "conversion": "B tensors unchanged; source masks stored as binary bool tensors; unused trained weights removed",
        }, indent=2))
    return destination


## 3. Indlæs model og konfiguration fra ZIP — ingen fortræning


In [ ]:
import datetime
import json
from ablation_experiment import load_bundle, run_ablation, make_plots, export_results, copy_export

BUNDLE = load_bundle(ZIP_PATH)
CONFIG = BUNDLE["config"]
SEEDS = [int(value.strip()) for value in TRAINING_SEEDS.split(",") if value.strip()]
if not SEEDS or len(SEEDS) != len(set(SEEDS)) or min(SEEDS) < 0:
    raise ValueError("TRAINING_SEEDS skal indeholde forskellige ikke-negative heltal, fx 0,1,2")
WORKERS = CONFIG.get("num_workers", 2) if NUM_WORKERS == -1 else NUM_WORKERS
if WORKERS < 0:
    raise ValueError("NUM_WORKERS skal være -1 eller ikke-negativ")
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
label = RUN_LABEL.strip() or f"ablation_{CONFIG['model_type']}_{CONFIG['dataset']}_{stamp}"
if label in (".", "..") or "/" in label or chr(92) in label:
    raise ValueError("RUN_LABEL skal være et mappenavn uden skråstreger")
RUN_OUTPUT = OUTPUT_ROOT / label
if RUN_OUTPUT.exists():
    raise FileExistsError("Resultatmappen findes allerede. Vælg et nyt RUN_LABEL; filer overskrives ikke.")
print("MODEL:", CONFIG["model_type"], "width=", CONFIG.get("width_multiplier", 1), "dataset=", CONFIG["dataset"])
print("KILDE: seed=", CONFIG["seed"], "k=", CONFIG["comparison"]["rewind_epoch"], "T=", CONFIG["comparison"]["total_epochs"])
print("SPARSITY:", CONFIG["pruning"]["sparsity"], "BATCH:", CONFIG["batch_size"])
print("FASER (samme optimizerindstillinger i begge betingelser):")
print(json.dumps(BUNDLE["plan"], indent=2))
print("Nye træningsepoker:", 2 * len(SEEDS) * sum(phase["epochs"] for phase in BUNDLE["plan"]))
print("Parrede træningsseeds:", SEEDS, "— samme kildemaske og B-checkpoint holdes faste.")
print("Ingen gentagelse af dense A/B eller A's maskesøgning.")
print("Resultater:", RUN_OUTPUT)


## 4. Kør de parrede ablationer


In [ ]:
import shutil
import platform
if not all(name in globals() for name in ("BUNDLE", "CONFIG", "RUN_OUTPUT", "SEEDS", "DEVICE", "CODE_DIR")):
    raise RuntimeError("Kør først afsnit 1–3, især efter en runtime-genstart.")
RUN_OUTPUT.mkdir(parents=True, exist_ok=False)
try:
    RESULTS = run_ablation(BUNDLE, RUN_OUTPUT, SEEDS, device=DEVICE,
                           dataset_root=Path("/content/growth-ablation/datasets"), num_workers=WORKERS)
finally:
    source_dir = RUN_OUTPUT / "source"
    source_dir.mkdir(exist_ok=True)
    for name in ['models.py', 'loaders.py', 'run_logging.py', 'utils.py', 'training.py', 'progressive.py', 'comparison.py', 'ablation_experiment.py']:
        shutil.copy2(CODE_DIR / name, source_dir / name)
    environment = {"python": platform.python_version(), "torch": torch.__version__,
                   "torchvision": torchvision.__version__, "numpy": np.__version__,
                   "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU test",
                   "zip_source": ZIP_SOURCE, "persistent_output": USE_PERSISTENT_OUTPUT}
    (RUN_OUTPUT / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")


## 5. Rapportfigur og gamle referencer


In [ ]:
SUMMARY = make_plots(RUN_OUTPUT)


## 6. ZIP-eksport og automatisk download


In [ ]:
ARCHIVE_PATH = export_results(RUN_OUTPUT, Path("/content"))
if USE_PERSISTENT_OUTPUT:
    DRIVE_ARCHIVE = copy_export(ARCHIVE_PATH, OUTPUT_ROOT)
    print("ZIP på Google Drive:", DRIVE_ARCHIVE)
print("ZIP:", ARCHIVE_PATH, f"({ARCHIVE_PATH.stat().st_size / 2**20:.1f} MiB)")
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(ARCHIVE_PATH))
